# Dimensional Modeling: Star Schema

##  Libs and Dataset

In [29]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
import pandas as pd

sc = SparkSession.builder.master('local[*]').getOrCreate()

In [30]:
sales_data = pd.read_csv("sales_data.csv")

## Modeling Star Schema

Functions

In [31]:
def create_dimension(df, columns, id_name):
    return (
        df[columns]
        .drop_duplicates()
        .reset_index(drop=True)
        .reset_index()
        .rename(columns={"index": id_name})
    )

def merge_and_replace(df, dim_df, keys):
    df = df.merge(dim_df, on=keys, how="inner")
    return df.drop(columns=keys)

Creating Dimensional Dataframes

In [32]:
users_df = create_dimension(
    sales_data, 
    ["user_name", "user_country_name", "user_city_name"], 
    "user_id")

products_df = create_dimension(
    sales_data, 
    ["product_name", "product_description", "product_brand", "product_value_in_dolars"], 
    "product_id")

stores_df = create_dimension(
    sales_data, 
    ["store_name", "store_country", "store_city"], 
    "store_id")


Merging with Fact Table

In [33]:
sales_data = merge_and_replace(
    sales_data, 
    users_df, 
    ["user_name", "user_country_name", "user_city_name"])

sales_data = merge_and_replace(
    sales_data, 
    products_df, 
    ["product_name", "product_description", "product_brand", "product_value_in_dolars"])

sales_data = merge_and_replace(
    sales_data, 
    stores_df, 
    ["store_name", "store_country", "store_city"])

sales_data = (
    sales_data
    .reset_index(drop=True)
    .reset_index()
    .rename(columns={"index": "id"})
)

## Data Analysis
_(With PySpark)_

In [34]:
SALES = sc.createDataFrame(sales_data)
PRODUCTS = sc.createDataFrame(products_df)
USERS = sc.createDataFrame(users_df)
STORES = sc.createDataFrame(stores_df)

Quais foram os países com maior número de vendas?

In [ ]:
country_sales_df = (
    SALES
    .join(
        STORES,
        "store_id",
        "left"
    )
    .groupBy("store_country")
    .agg(
        F.count(F.col("id")).alias("sales_total")
    )
    .orderBy(F.col("sales_total").desc())
)

country_sales_df.show()

+-------------+-----------+
|store_country|sales_total|
+-------------+-----------+
|       Canada|      13035|
|    Australia|      12469|
|          USA|      12271|
|           UK|      12225|
+-------------+-----------+



Quais países tem mais lojas ?

In [ ]:
stores_per_contry_df = (
    STORES
    .groupBy("store_country")
    .agg(
        F.count(F.col("store_id")).alias("stores_total")
    )
    .orderBy(F.col("stores_total").desc())
  )

stores_per_contry_df.show()

+-------------+------------+
|store_country|stores_total|
+-------------+------------+
|       Canada|          18|
|          USA|          17|
|           UK|          17|
|    Australia|          17|
+-------------+------------+



Quais países tiveram o maior valor total bruto de vendas ?

In [ ]:
sales_per_store = (
  SALES
  .join(
      PRODUCTS
      .select("product_id", "product_value_in_dolars"),
      "product_id",
      "left"
  )
  .withColumn("paid_value", F.col("product_value_in_dolars") * F.col("buyed_amount"))
  .join(
      STORES
      .select("store_id", "store_country", "store_name"),
      "store_id",
      "left"
  )
)

sales_total_value = (
    sales_per_store
    .groupBy("store_country")
    .agg(
        F.sum(F.col("paid_value")).alias("total_value")
    )
    .orderBy(F.col("total_value").desc())
)

sales_total_value.show()

+-------------+-----------------+
|store_country|      total_value|
+-------------+-----------------+
|       Canada|7246033.970000941|
|    Australia|6896470.250000843|
|          USA|6860512.980000877|
|           UK|6808784.100000873|
+-------------+-----------------+



Quais são as maiores lojas em número de vendas de cada país ?

In [ ]:
windowSpec = Window.partitionBy("store_country").orderBy(F.col("total_value").desc())

sales_total_value = (
    sales_per_store
    .groupBy("store_name", "store_country")
    .agg(
        F.sum(F.col("paid_value")).alias("total_value")
    )
    .orderBy(F.col("total_value").desc())
    .withColumn("rank", F.rank().over(windowSpec))
    .filter(F.col("rank") == 1)
    .drop("rank")
)

sales_total_value.show()

+-----------------+-------------+------------------+
|       store_name|store_country|       total_value|
+-----------------+-------------+------------------+
|            Kmart|    Australia| 796322.1199999846|
|    Canadian Tire|       Canada|1255490.5799999647|
|              B&Q|           UK|1231429.1599999631|
|Fry's Electronics|          USA| 839137.0199999824|
+-----------------+-------------+------------------+



Quais foram as lojas com maior valor bruto de venda ?

In [ ]:
sales_total_value = (
    sales_per_store
    .groupBy("store_name")
    .agg(
        F.sum(F.col("paid_value")).alias("total_value")
    )
    .orderBy(F.col("total_value").desc())
)

sales_total_value.show()

+------------------+------------------+
|        store_name|       total_value|
+------------------+------------------+
|     Canadian Tire|1255490.5799999647|
|               B&Q|1231429.1599999631|
|Shoppers Drug Mart| 908500.2699999809|
| Fry's Electronics| 839137.0199999824|
|             Argos| 827542.0399999847|
|        John Lewis| 825027.0799999841|
|             Sears| 819131.6199999821|
|             Kmart| 796322.1199999846|
|        Home Depot| 795201.7699999855|
|           Loblaws| 788921.6099999845|
|       Officeworks| 783131.9899999851|
|             Metro|  764608.509999987|
|     Harvey Norman| 725822.6899999871|
|       David Jones|459832.89999999845|
|          JB Hi-Fi| 458423.0199999981|
|          GameStop| 454511.7299999996|
|            Target|445747.13999999955|
|      Office Depot|445741.59999999875|
|         Sams Club| 436719.1299999991|
|          Halfords| 433828.8099999989|
+------------------+------------------+
only showing top 20 rows

